In [1]:
import os
import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch import nn
import torchvision
from sklearn.model_selection import train_test_split
import pandas as pd

In [2]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

In [3]:
set_seed(42)

# Dataset Prepare

In [4]:
keypoints_dir = '/kaggle/input/datasets/ntnhan2005/signlanguagedetection-processed/keypoints-20261001T071139Z-1-001/keypoints'
keypoints_paths = []
labels = []
labels_csv = pd.read_csv('/kaggle/input/datasets/aresusayhi/vsl-vietnamese-sign-languages/Dataset/Labels/label.csv')

for filename in os.listdir(keypoints_dir):
    keypoints_paths.append(os.path.join(keypoints_dir, filename))
    video_name = filename.split('.')[0] + '.mp4'
    label = labels_csv.loc[labels_csv['VIDEO'] == video_name, 'LABEL'].iloc[0]
    labels.append(label)
print(len(keypoints_paths))
print(len(labels))
print(labels[0])

4362
4362
con cua


In [5]:
all_labels = set(labels)

mapping_labels = {label : i for i, label in enumerate(all_labels)}


In [6]:
class KeypointDataset(Dataset):
    def __init__(self, keypoints_paths, labels, mapping_labels):
        self.keypoints_paths = keypoints_paths
        self.labels = labels
        self.mapping_labels = mapping_labels

    def __len__(self):
        return len(self.keypoints_paths)

    def __getitem__(self, index):
        keypoints = np.load(self.keypoints_paths[index])
        keypoints = torch.tensor(keypoints)
        return keypoints, self.mapping_labels[self.labels[index]]

In [7]:
from collections import Counter
counter= Counter(labels)

In [8]:
print(len(all_labels))

3315


In [9]:
one_video_paths, varius_video_paths = [], []
one_video_labels, varius_video_labels = [], []
for i in range(len(labels)):
    if (counter[labels[i]] == 1):
        one_video_paths.append(keypoints_paths[i])
        one_video_labels.append(labels[i])
    else:
        varius_video_paths.append(keypoints_paths[i])
        varius_video_labels.append(labels[i])

In [10]:
train_paths, val_paths, train_labels, val_labels = train_test_split(varius_video_paths, varius_video_labels, test_size = 0.2, random_state = 0)

In [11]:
train_paths.extend(one_video_paths)
train_labels.extend(one_video_labels)

In [12]:
train_dataset = KeypointDataset(train_paths, train_labels, mapping_labels)
val_dataset = KeypointDataset(val_paths, val_labels, mapping_labels)

In [13]:
from torch.nn.utils.rnn import pad_sequence
def collate_fn(batch):
    frames = [item[0] for item in batch]
    labels = torch.tensor([item[1] for item in batch])

    frames = [
        x.reshape(x.shape[0], -1)
        for x in frames
    ]

    frames = pad_sequence(
        frames,
        batch_first=True,
        padding_value=0
    )

    return frames, labels

In [14]:
train_loader = DataLoader(train_dataset, batch_size = 16, shuffle = True, collate_fn = collate_fn)
val_loader = DataLoader(val_dataset, batch_size = 16, shuffle = False, collate_fn = collate_fn)

# Model

In [15]:
class BaselineModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super().__init__()
        self.lstm = nn.LSTM(input_size = input_size, hidden_size = hidden_size, num_layers = num_layers, batch_first = True)
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, inputs):
        outputs, (h_n, c_n) = self.lstm(inputs)
        last_hidden = h_n[-1]
        outputs = self.classifier(last_hidden)
        return outputs

# Train and Evaluate

In [17]:
from tqdm import tqdm
from sklearn.metrics import f1_score

set_seed(42)

device = 'cuda'
model = BaselineModel(input_size = 126, hidden_size = 256, num_layers = 2, num_classes = len(all_labels)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr = 1e-4, weight_decay = 1e-5)
loss_fn = nn.CrossEntropyLoss()


#----- Config -----#
num_epochs = 100
best_f1 = -1.0
patience = 10
cnt = 0

for epoch in range(1, num_epochs + 1):
    if cnt >= patience:
        break
    model.train()
    train_loss = 0.0
    train_gt = []
    train_pred = []
    for inputs, labels in tqdm(train_loader):
        optimizer.zero_grad()
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        outputs = model(inputs)

        loss = loss_fn(outputs, labels)

        loss.backward()
        optimizer.step()

        outputs = outputs.argmax(dim = 1)
        train_gt.extend(labels.cpu().numpy())
        train_pred.extend(outputs.cpu().numpy())

        train_loss += loss.item()

    train_loss = train_loss / (len(train_loader))
    train_f1 = f1_score(train_gt, train_pred, average = 'macro')

    model.eval()
    val_loss = 0.0
    val_gt = []
    val_pred = []
    with torch.inference_mode():
        for inputs, labels in tqdm(val_loader):
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            outputs = model(inputs)
    
            loss = loss_fn(outputs, labels)
            
            outputs = outputs.argmax(dim = 1)
            val_gt.extend(labels.cpu().numpy())
            val_pred.extend(outputs.cpu().numpy())
    
            val_loss += loss.item()

        val_loss = val_loss / (len(val_loader))
        val_f1 = f1_score(val_gt, val_pred, average = 'macro')

    if best_f1 < val_f1:
        best_f1 = val_f1
        cnt = 0
        torch.save(model.state_dict(), f = 'best_model.pth')
    else:
        cnt += 1
    
    print(f'Epoch {epoch}/{num_epochs} | Train Loss: {train_loss:.4f} | Train F1: {train_f1:.4f} | Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f}')

100%|██████████| 20/20 [00:00<00:00, 58.92it/s]


Epoch 1/100 | Train Loss: 8.1133 | Train F1: 0.0000 | Val Loss: 8.1448 | Val F1: 0.0009


100%|██████████| 20/20 [00:00<00:00, 58.05it/s]


Epoch 2/100 | Train Loss: 8.1047 | Train F1: 0.0000 | Val Loss: 8.0943 | Val F1: 0.0000


100%|██████████| 20/20 [00:00<00:00, 56.98it/s]


...

Epoch 98/100 | Train Loss: 0.7704 | Train F1: 0.9120 | Val Loss: 6.6658 | Val F1: 0.2115


100%|██████████| 20/20 [00:00<00:00, 55.21it/s]


Epoch 99/100 | Train Loss: 0.7480 | Train F1: 0.9214 | Val Loss: 6.6417 | Val F1: 0.2001


100%|██████████| 20/20 [00:00<00:00, 59.51it/s]

Epoch 100/100 | Train Loss: 0.7422 | Train F1: 0.9203 | Val Loss: 6.6739 | Val F1: 0.2140


In [18]:
model.load_state_dict(torch.load('/kaggle/working/best_model.pth'))

<All keys matched successfully>

In [19]:
from sklearn.metrics import classification_report

model.eval()
val_loss = 0.0
val_gt = []
val_pred = []
with torch.inference_mode():
    for inputs, labels in tqdm(val_loader):
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        outputs = model(inputs)

        loss = loss_fn(outputs, labels)
        
        outputs = outputs.argmax(dim = 1)
        val_gt.extend(labels.cpu().numpy())
        val_pred.extend(outputs.cpu().numpy())

        val_loss += loss.item()

    val_loss = val_loss / (len(val_loader))
    val_f1 = f1_score(val_gt, val_pred, average = 'macro')


print(classification_report(val_gt, val_pred))

100%|██████████| 20/20 [00:00<00:00, 63.13it/s]

              precision    recall  f1-score   support

           1       0.00      0.00      0.00         1
           4       0.00      0.00      0.00         1
          13       0.00      0.00      0.00         1
          15       0.00      0.00      0.00         0
          19       0.00      0.00      0.00         1
          20       0.00      0.00      0.00         0
          33       0.00      0.00      0.00         0
          38       0.00      0.00      0.00         1
          60       1.00      0.50      0.67         2
          83       1.00      1.00      1.00         1
          87       0.50      1.00      0.67         1
          99       0.00      0.00      0.00         0
         100       0.00      0.00      0.00         0
         105       0.00      0.00      0.00         0
         116       0.00      0.00      0.00         1
         122       1.00      1.00      1.00         1
         150       0.00      0.00      0.00         0
         158       0.00    


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/